# GPT-2 355M Function Calling — Full Test-Split Evaluation

Installs the [`gpt2fc` package](https://github.com/mron03/gpt2-function-calling) straight from GitHub, downloads the fine-tuned checkpoint from the [Hugging Face model repo](https://huggingface.co/noFFENSE/gpt2-355M-function-calling), and runs `gpt2fc-eval` on the **full held-out test split** (last 5% of Glaive Function Calling v2, ~5,648 samples — the same split arithmetic training used, so none of it was seen).

**Before running:**
- Accelerator → **GPU T4 x2** (Kaggle) or **T4 GPU** (Colab)
- Internet → **On**

Decoding is KV-cached greedy, one sample at a time — expect roughly **2–3 hours** on a T4 for the full split. Set `NUM_SAMPLES` to something small first if you just want a smoke test.

In [ ]:
!pip install -q git+https://github.com/mron03/gpt2-function-calling

import torch
print('CUDA:', torch.cuda.is_available(), '—', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'enable the GPU accelerator!')

In [ ]:
from huggingface_hub import hf_hub_download

CHECKPOINT = hf_hub_download(
    repo_id='noFFENSE/gpt2-355M-function-calling',
    filename='gpt2-355M-function-calling.pth',
)
print('Checkpoint:', CHECKPOINT)

In [ ]:
NUM_SAMPLES    = -1     # -1 = full test split (~5,648 samples); use e.g. 20 for a smoke test
MAX_NEW_TOKENS = 256
OUTPUT_JSON    = 'eval-full-355M.json'

!gpt2fc-eval \
    --checkpoint {CHECKPOINT} \
    --model-size 355M \
    --num-samples {NUM_SAMPLES} \
    --max-new-tokens {MAX_NEW_TOKENS} \
    --output-json {OUTPUT_JSON}

In [ ]:
import json
import pandas as pd

with open(OUTPUT_JSON) as f:
    results = json.load(f)

print('=== Metrics ===')
print(json.dumps(results['metrics'], indent=2))

print('\n=== Sample predictions (first 20) ===')
df = pd.DataFrame(results['samples'][:20])[['ground_truth', 'prediction', 'gt_fc', 'pred_fc']]
pd.set_option('display.max_colwidth', 120)
display(df)

On Kaggle the results JSON lands in `/kaggle/working` — download it from the **Output** tab (it includes every prompt, prediction, and parsed call, so failures can be inspected offline).